## 01-EDA Add MMSE scores to split files



### Import data

In [94]:
# import required packages
import pandas as pd
import os

In [95]:
# define paths
inputs_dir_path = "../data/splits/"
outputs_dir_path = "../data/splits/outputs/"
mmse_scores_fname = "../data/Training Dataset/Diagnosis training data/ADReSSo21/diagnosis/train/adresso-train-mmse-scores.csv"
pitt_mmse_scores_fname = "../data/TalkBank downloads/Pitt/Pitt-data.xlsx"

In [96]:
# import split csv files as dataframes
data_dict = {
    f.split(".")[0]: pd.read_csv(inputs_dir_path + f)
    for f in os.listdir(inputs_dir_path)
    if f.endswith(".csv")
}
data_dict

{'train_50pc':      file_id                                               text target
 0   adrso021  Everything you see going on in this picture, g...     CN
 1   adrso265  Now you just look at the picture. Okay, this i...     CN
 2   adrso237  Tell me everything that you see going on in th...     AD
 3   adrso197  Girls and boys getting her one that stores sto...     AD
 4   adrso017  All right, we have a few more tasks to go thro...     CN
 ..       ...                                                ...    ...
 78  adrso298  Little Girl is reaching for her brother to giv...     CN
 79  adrso072  Tell me what's going on. Well the kids have ta...     AD
 80  adrso070  Okay what do you see happening in that picture...     AD
 81  adrso178  I want you to do to look at this picture and t...     CN
 82  adrso246  Tell me what you see going on in the picture. ...     AD
 
 [83 rows x 3 columns],
 'train_pitt':     file_id                                               text target
 0     001-

In [97]:
# import mmse files
mmse_df = pd.read_csv(mmse_scores_fname, usecols=["adressfname", "mmse", "dx"])
pitt_mmse_df = pd.read_excel(
    pitt_mmse_scores_fname,
    sheet_name="data",
    header=2,
    usecols=["id", "mms", "mmse2", "mmse3", "mmse4", "mmse5"],
)

/Users/la/miniconda3/envs/mds/lib/python3.11/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)


### Clean Pitt MMSE data

Note: in the readme tab of the Pitt spreadsheet, it states that:

"The number following the dash corresponds to the visit number: 0 (baseline), 1, 2, 3, and 4.  On the Data Spreadsheet, these numbers are entered as 1 (baseline), 2, 3, 4, and 5.  So, 001-0.cha corresponds to visit 1 data on the spreadsheet for id #1; 001-1.cha corresponds to visit 2 data on the spreadsheet for id #1, etc."

In [98]:
pitt_mmse_df = pitt_mmse_df.rename(columns={"mms": "mmse1"})
pitt_mmse_df = pd.wide_to_long(
    pitt_mmse_df, i="id", j="interview_number", stubnames="mmse"
).reset_index()
pitt_mmse_df["id"] = pitt_mmse_df["id"].astype(str).str.pad(width=3, fillchar="0")
pitt_mmse_df["interview_number"] = pitt_mmse_df["interview_number"] - 1
pitt_mmse_df["id"] = (
    pitt_mmse_df["id"] + "-" + pitt_mmse_df["interview_number"].astype(str)
)
pitt_mmse_df = pitt_mmse_df[["id", "mmse"]]
pitt_mmse_df = pitt_mmse_df[~pitt_mmse_df["mmse"].isnull()]
pitt_mmse_df

,id,mmse
0,001-0,18.0
1,002-0,30.0
2,003-0,20.0
3,005-0,23.0
4,006-0,28.0
...,...,...
2470,621-4,10.0
2479,631-4,28.0
2502,663-4,19.0
2516,681-4,19.0


In [99]:
# merge data and export to csv
for file, df in data_dict.items():
    if "test" in file:
        pass
    elif "pitt" in file:
        data_dict[file] = df.merge(
            pitt_mmse_df, left_on="file_id", right_on="id", how="left"
        )[["file_id", "target", "text", "mmse"]]
    else:
        data_dict[file] = df.merge(mmse_df, left_on="file_id", right_on="adressfname")[
            ["file_id", "text", "target", "mmse"]
        ]
    print(f"Features in {file} have null rows: {data_dict[file].isna().sum()}")
    data_dict[file].to_csv(outputs_dir_path + file + ".csv", index=False)

Features in train_50pc have null rows: file_id    0
text       0
target     0
mmse       0
dtype: int64
Features in train_pitt have null rows: file_id     0
target      0
text        0
mmse       91
dtype: int64
Features in test_text have null rows: ID             0
Prediction    71
text           0
dtype: int64
Features in dev_50pc have null rows: file_id    0
text       0
target     0
mmse       0
dtype: int64


Note: there are files with no corresponding matches in the Pitt spreadsheet but there are also visit data in the spreadsheet with no corresponding files